## Google Maps Places (New) API

See [here](https://developers.google.com/maps/documentation/places/web-service/text-search#includedtype) for example requests using Web SDK.

#### Pilot Extraction
- Select 10 LADs (based on top & bottom 3 by population, and top & bottom 2 by land size)
- Prep polygon geomtries from geojson files
- Split polygons where necessary

#### Full Extraction
- Repeat for rest of LADs

In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import geopandas as gpd
import json
from tqdm import tqdm
tqdm.pandas()

### Select 10 LADs for Pilot

In [2]:
from data.ukps_data_dict import lad23cd_codes

ukps_df = pd.read_csv('../data/UKDA-9350-tab/tab/participation_2023-24_annual_data_safeguard.tab', sep="\t")
ukps_df['LAD23CD'] = ukps_df.lad23cd.map(lad23cd_codes)
ukps_lad_set = set(ukps_df['LAD23CD'])

In [3]:
oa_lad_map = pd.read_excel("../data/sapeoatablefinal2022v2.xlsx", sheet_name="Mid-2022 OA 2021", skiprows=3)
oa_lad_map = oa_lad_map.loc[oa_lad_map['LAD 2021 Code'].isin(ukps_lad_set)]
shortlisted_lads = oa_lad_map.groupby('LAD 2021 Code').agg({
    'LAD 2021 Name':pd.Series.unique,
    'Total': sum
})
shortlisted_lads = pd.concat([
    shortlisted_lads.sort_values("Total").head(3),
    shortlisted_lads.sort_values("Total").tail(3)
])
shortlisted_lads

/var/folders/rw/x_vg479j72xgn55l7gcx378r0000gn/T/ipykernel_40616/3830180970.py:3: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  shortlisted_lads = oa_lad_map.groupby('LAD 2021 Code').agg({


,LAD 2021 Name,Total
LAD 2021 Code,,
E06000053,[Isles of Scilly],2281
E09000001,[City of London],10847
E06000017,[Rutland],41151
E06000052,[Cornwall],575413
E08000035,[Leeds],822483
E08000025,[Birmingham],1157603


In [4]:
with open('../data/Local_Authority_Districts_December_2023_Boundaries_UK_BFE_7168133065712352501.geojson') as f:
    bounds_gjs = json.load(f)
bounds_gdf = gpd.GeoDataFrame.from_features(bounds_gjs.get('features'))
bounds_gdf = bounds_gdf.loc[bounds_gdf['LAD23CD'].isin(ukps_lad_set)]
bounds_gdf['area'] = bounds_gdf['geometry'].area
bounds_gdf.head()

,geometry,FID,LAD23CD,LAD23NM,LAD23NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,area
0,"POLYGON ((-1.26846 54.72612, -1.26822 54.72609...",1,E06000001,Hartlepool,,447160,531474,-1.27018,54.67614,5de18a56-bb08-4d29-82cb-662fdb90d44a,0.013703
1,"POLYGON ((-1.25112 54.59153, -1.24953 54.59151...",2,E06000002,Middlesbrough,,451141,516887,-1.21099,54.54467,49af2d4f-59b0-49f9-b537-33b217c1537d,0.007577
2,"POLYGON ((-1.14105 54.64773, -1.13798 54.64737...",3,E06000003,Redcar and Cleveland,,464361,519597,-1.00608,54.56752,b99c3451-43c1-46d8-b649-ee168b402a68,0.035260
3,"POLYGON ((-1.31729 54.6448, -1.31715 54.6448, ...",4,E06000004,Stockton-on-Tees,,444940,518179,-1.30664,54.55687,b2f28574-2432-4cc8-aac3-3de359df13d3,0.029147
4,"POLYGON ((-1.63768 54.61714, -1.63767 54.6167,...",5,E06000005,Darlington,,428029,515648,-1.56835,54.53534,936dca9c-faa1-4aca-ad1c-fd81b92fd03d,0.027436


In [9]:
test_area = bounds_gdf.loc[bounds_gdf.LAD23CD.isin(shortlisted_lads.index)]
test_area

,geometry,FID,LAD23CD,LAD23NM,LAD23NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,area
16,"POLYGON ((-0.60944 52.75973, -0.60909 52.7597,...",17,E06000017,Rutland,,492992,308655,-0.62630,52.66765,4403b9ad-da3a-4240-9c9e-a396071bc4b0,0.052322
48,"MULTIPOLYGON (((-5.26694 50.01356, -5.26695 50...",49,E06000052,Cornwall,,212497,64493,-4.64254,50.45022,9bc220b9-8378-4c34-8178-6b1652649446,0.456646
49,"MULTIPOLYGON (((-6.39885 49.86569, -6.39874 49...",50,E06000053,Isles of Scilly,,91327,11447,-6.30217,49.92332,afca461c-801c-4df1-b0f1-167b46294e75,0.002857
250,"POLYGON ((-1.82482 52.60778, -1.82374 52.60747...",251,E08000025,Birmingham,,408150,287352,-1.88141,52.48404,06b4cfe9-de02-434d-9afa-4186e208111e,0.035453
260,"POLYGON ((-1.34057 53.94488, -1.34074 53.94468...",261,E08000035,Leeds,,432528,436384,-1.50736,53.82273,25f8a824-ef47-4eda-aabe-1de93fb85053,0.075328
263,"POLYGON ((-0.09669 51.52319, -0.09668 51.52317...",264,E09000001,City of London,,532382,181358,-0.09351,51.51564,3423eef3-091e-4798-aad0-338669589b08,0.000408


In [10]:
m = test_area.set_crs("EPSG:4326").explore(
    tiles="OpenStreetMap",
    tooltip='LAD23NM'
)
m

In [11]:
test_area['geom_split'] = test_area.geometry.apply(lambda x: list(x.geoms) if x.geom_type=='MultiPolygon' else [x])
test_area = test_area.explode("geom_split")
test_area

/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,geometry,FID,LAD23CD,LAD23NM,LAD23NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,area,geom_split
16,"POLYGON ((-0.60944 52.75973, -0.60909 52.7597,...",17,E06000017,Rutland,,492992,308655,-0.62630,52.66765,4403b9ad-da3a-4240-9c9e-a396071bc4b0,0.052322,"POLYGON ((-0.609443616633852 52.7597305919592,..."
48,"MULTIPOLYGON (((-5.26694 50.01356, -5.26695 50...",49,E06000052,Cornwall,,212497,64493,-4.64254,50.45022,9bc220b9-8378-4c34-8178-6b1652649446,0.456646,"POLYGON ((-5.26694003341016 50.0135639518638, ..."
48,"MULTIPOLYGON (((-5.26694 50.01356, -5.26695 50...",49,E06000052,Cornwall,,212497,64493,-4.64254,50.45022,9bc220b9-8378-4c34-8178-6b1652649446,0.456646,"POLYGON ((-5.74195180461415 50.0642116161264, ..."
48,"MULTIPOLYGON (((-5.26694 50.01356, -5.26695 50...",49,E06000052,Cornwall,,212497,64493,-4.64254,50.45022,9bc220b9-8378-4c34-8178-6b1652649446,0.456646,"POLYGON ((-5.7442822562866 50.0658915722518, -..."
48,"MULTIPOLYGON (((-5.26694 50.01356, -5.26695 50...",49,E06000052,Cornwall,,212497,64493,-4.64254,50.45022,9bc220b9-8378-4c34-8178-6b1652649446,0.456646,"POLYGON ((-5.74605471213743 50.0668734934518, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
49,"MULTIPOLYGON (((-6.39885 49.86569, -6.39874 49...",50,E06000053,Isles of Scilly,,91327,11447,-6.30217,49.92332,afca461c-801c-4df1-b0f1-167b46294e75,0.002857,"POLYGON ((-6.32349763443138 49.9798983592307, ..."
49,"MULTIPOLYGON (((-6.39885 49.86569, -6.39874 49...",50,E06000053,Isles of Scilly,,91327,11447,-6.30217,49.92332,afca461c-801c-4df1-b0f1-167b46294e75,0.002857,"POLYGON ((-6.29495232396935 49.9813557260253, ..."
250,"POLYGON ((-1.82482 52.60778, -1.82374 52.60747...",251,E08000025,Birmingham,,408150,287352,-1.88141,52.48404,06b4cfe9-de02-434d-9afa-4186e208111e,0.035453,"POLYGON ((-1.82482002022059 52.6077840088598, ..."
260,"POLYGON ((-1.34057 53.94488, -1.34074 53.94468...",261,E08000035,Leeds,,432528,436384,-1.50736,53.82273,25f8a824-ef47-4eda-aabe-1de93fb85053,0.075328,"POLYGON ((-1.340565695099 53.9448842475508, -1..."


In [ ]:
# test_area.to_file("../data/pilot_lad.geojson")

In [12]:
rest_of_uk = bounds_gdf.loc[~bounds_gdf.LAD23CD.isin(shortlisted_lads.index)]
rest_of_uk['geom_split'] = rest_of_uk.geometry.apply(lambda x: list(x.geoms) if x.geom_type=='MultiPolygon' else [x])
rest_of_uk = rest_of_uk.explode("geom_split")
rest_of_uk

/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,geometry,FID,LAD23CD,LAD23NM,LAD23NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,area,geom_split
0,"POLYGON ((-1.26846 54.72612, -1.26822 54.72609...",1,E06000001,Hartlepool,,447160,531474,-1.27018,54.67614,5de18a56-bb08-4d29-82cb-662fdb90d44a,0.013703,"POLYGON ((-1.26845558516825 54.7261163520838, ..."
1,"POLYGON ((-1.25112 54.59153, -1.24953 54.59151...",2,E06000002,Middlesbrough,,451141,516887,-1.21099,54.54467,49af2d4f-59b0-49f9-b537-33b217c1537d,0.007577,"POLYGON ((-1.25112183522325 54.5915298068587, ..."
2,"POLYGON ((-1.14105 54.64773, -1.13798 54.64737...",3,E06000003,Redcar and Cleveland,,464361,519597,-1.00608,54.56752,b99c3451-43c1-46d8-b649-ee168b402a68,0.035260,"POLYGON ((-1.14105025960825 54.6477294490761, ..."
3,"POLYGON ((-1.31729 54.6448, -1.31715 54.6448, ...",4,E06000004,Stockton-on-Tees,,444940,518179,-1.30664,54.55687,b2f28574-2432-4cc8-aac3-3de359df13d3,0.029147,"POLYGON ((-1.31728586216906 54.6448033265433, ..."
4,"POLYGON ((-1.63768 54.61714, -1.63767 54.6167,...",5,E06000005,Darlington,,428029,515648,-1.56835,54.53534,936dca9c-faa1-4aca-ad1c-fd81b92fd03d,0.027436,"POLYGON ((-1.6376776536223 54.6171377919482, -..."
...,...,...,...,...,...,...,...,...,...,...,...,...
291,"POLYGON ((-0.17473 51.39337, -0.17471 51.39336...",292,E09000029,Sutton,,527357,163639,-0.17226,51.35755,47749f83-9731-416f-8648-9cd80ce1da45,0.005660,"POLYGON ((-0.17473424784392 51.393372565324, -..."
292,"POLYGON ((-0.02903 51.54235, -0.02901 51.54232...",293,E09000030,Tower Hamlets,,536340,181452,-0.03647,51.51554,b148b7fd-6e47-4778-95c1-4bee9c3d39e9,0.002795,POLYGON ((-0.0290280341039871 51.5423547092921...
293,"POLYGON ((-0.00798 51.64632, -0.0076 51.64622,...",294,E09000031,Waltham Forest,,537328,190278,-0.01880,51.59461,aba8cc07-733f-4f60-a128-b9e71501c93d,0.005035,POLYGON ((-0.0079769748621148 51.6463243800225...
294,"POLYGON ((-0.12825 51.48485, -0.12815 51.48473...",295,E09000032,Wandsworth,,525152,174138,-0.20021,51.45240,463d5802-96f9-4e76-95cc-fd945272f3c0,0.004555,"POLYGON ((-0.128254117493704 51.4848497919891,..."


### Send Place TextSearch API requests

In [13]:
from scripts.cf_typology import gmaps_types
from scripts.google_places_search import PlaceTextSearchClient
from datetime import datetime

print(gmaps_types.keys())

save_places_dir = "../data/place_ids"
os.makedirs(save_places_dir, exist_ok=True)

now_ts = datetime.now().strftime("%d-%m-%Y_%H%M%S") #28-02-2026_101817 pilot || #28-02-2026_121413 rest_of_uk 
                                                    #23-04-2026_163244 investigation || #23-04-2026_204939 re-run all areas with splits
print(now_ts)

dict_keys(['heritage', 'museum', 'reading', 'cinema', 'theatre', 'nightlife', 'hybrid'])
02-05-2026_141057


In [14]:
from scripts.google_place_agg import _split_polygon
import math

full_uk = pd.concat([test_area, rest_of_uk]).reset_index(drop=True)
full_uk['area'] = gpd.GeoDataFrame(full_uk, geometry='geom_split')['geom_split'].area
full_uk['split'] = full_uk['area'].apply( lambda x: max(0, math.ceil( math.log(x / 0.05) / math.log(4) )) )
print( full_uk['split'].value_counts() )

polygon_list = []
for _,r in full_uk.iterrows():
    if r.split <=0:
        polygon_list.append( r.geom_split )
    else:
        to_split = [r.geom_split]
        for i in range(r.split):
            split_i = []
            while len(to_split)>0:
                split_i.extend(_split_polygon(to_split.pop(-1)))
            to_split = split_i.copy()
        after_split = split_i.copy()
        polygon_list.extend( after_split )
display( pd.Series([x.area for x in polygon_list]).describe() )

print(len(polygon_list))

split
0    294
1     80
2     14
3      1
Name: count, dtype: int64


/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/shapely/set_operations.py:451: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


count    1.113000e+03
mean     1.585054e-02
std      1.428510e-02
min      1.504188e-11
25%      5.420251e-04
50%      1.525209e-02
75%      2.384061e-02
max      7.090806e-02
dtype: float64

1113


In [ ]:
include_types = gmaps_types['museum']
print(include_types)

museum_c = PlaceTextSearchClient()
museum_pids = set()
for query in include_types:
    print(f"########### Starting search for: {query} ################")
    for poly in tqdm(polygon_list):
        museum_pids = museum_pids.union(
        museum_c.paginate_requests_placeid_only(
            query.replace("_", " "), poly, page_token=None, included_type=query)
        )
    print(len(museum_pids))

pd.DataFrame({"place_ids": list(museum_pids)}).assign(place_type="museum").to_parquet(f"{save_places_dir}/{now_ts}_museum.parquet")

['planetarium', 'wildlife_park', 'zoo', 'art_museum']
########### Starting search for: planetarium ################


100%|██████████| 1113/1113 [04:12<00:00,  4.40it/s]


7610
########### Starting search for: wildlife_park ################


100%|██████████| 1113/1113 [04:22<00:00,  4.24it/s]


7762
########### Starting search for: zoo ################


100%|██████████| 1113/1113 [04:24<00:00,  4.21it/s]


8125
########### Starting search for: art_museum ################


100%|██████████| 1113/1113 [04:41<00:00,  3.95it/s]


8137


In [19]:
include_types = gmaps_types['heritage']
print(include_types)

heritage_c = PlaceTextSearchClient()
heritage_pids = set()
for query in include_types:
    print(f"########### Starting search for: {query} ################")
    for poly in tqdm(polygon_list):
        heritage_pids = heritage_pids.union(
        heritage_c.paginate_requests_placeid_only(
            query.replace("_", " "), poly, page_token=None, included_type=query)
        )
    print(len(heritage_pids))

pd.DataFrame({"place_ids": list(heritage_pids)}).assign(place_type="heritage").to_parquet(f"{save_places_dir}/{now_ts}_heritage.parquet")

['cultural_landmark', 'historical_landmark', 'historical_place', 'monument', 'sculpture', 'tourist_attraction', 'history_museum', 'fountain', 'castle']
########### Starting search for: cultural_landmark ################


100%|██████████| 1113/1113 [04:44<00:00,  3.92it/s]


407
########### Starting search for: historical_landmark ################


100%|██████████| 1113/1113 [07:32<00:00,  2.46it/s]


14817
########### Starting search for: historical_place ################


...

31151
########### Starting search for: fountain ################


100%|██████████| 1113/1113 [04:06<00:00,  4.51it/s]


31174
########### Starting search for: castle ################


100%|██████████| 1113/1113 [04:26<00:00,  4.18it/s]

31240


In [20]:
include_types = gmaps_types['theatre']
print(include_types)

theatre_c = PlaceTextSearchClient()
theatre_pids = set()
for query in include_types:
    print(f"########### Starting search for: {query} ################")
    for poly in tqdm(polygon_list):
        theatre_pids = theatre_pids.union(
        theatre_c.paginate_requests_placeid_only(
            query.replace("_", " "), poly, page_token=None, included_type=query)
        )
    print(len(theatre_pids))

pd.DataFrame({"place_ids": list(theatre_pids)}).assign(place_type="theatre").to_parquet(f"{save_places_dir}/{now_ts}_theatre.parquet")

['amphitheatre', 'auditorium', 'opera_house', 'performing_arts_theater']
########### Starting search for: amphitheatre ################


100%|██████████| 1113/1113 [03:38<00:00,  5.08it/s]


46
########### Starting search for: auditorium ################


100%|██████████| 1113/1113 [03:47<00:00,  4.89it/s]


414
########### Starting search for: opera_house ################


100%|██████████| 1113/1113 [04:26<00:00,  4.18it/s]


415
########### Starting search for: performing_arts_theater ################


100%|██████████| 1113/1113 [04:56<00:00,  3.76it/s]

2470


In [21]:
include_types = gmaps_types['reading']
print(include_types)

reading_c = PlaceTextSearchClient()
reading_pids = set()
for query in include_types:
    print(f"########### Starting search for: {query} ################")
    for poly in tqdm(polygon_list):
        reading_pids = reading_pids.union(
        reading_c.paginate_requests_placeid_only(
            query.replace("_", " "), poly, page_token=None, included_type=query)
        )
    print(len(reading_pids))

pd.DataFrame({"place_ids": list(reading_pids)}).assign(place_type="reading").to_parquet(f"{save_places_dir}/{now_ts}_reading.parquet")


['library', 'book_store']
########### Starting search for: library ################


100%|██████████| 1113/1113 [05:04<00:00,  3.65it/s]


4379
########### Starting search for: book_store ################


100%|██████████| 1113/1113 [05:43<00:00,  3.24it/s]

9723


In [22]:
include_types = gmaps_types['cinema']
print(include_types)

cinema_c = PlaceTextSearchClient()
cinema_pids = set()
for query in include_types:
    print(f"########### Starting search for: {query} ################")
    for poly in tqdm(polygon_list):
        cinema_pids = cinema_pids.union(
        cinema_c.paginate_requests_placeid_only(
            query.replace("_", " "), poly, page_token=None, included_type=query)
        )
    print(len(cinema_pids))

pd.DataFrame({"place_ids": list(cinema_pids)}).assign(place_type="cinema").to_parquet(f"{save_places_dir}/{now_ts}_cinema.parquet")

['movie_theater']
########### Starting search for: movie_theater ################


100%|██████████| 1113/1113 [04:50<00:00,  3.83it/s]

1091


In [24]:
include_types = gmaps_types['nightlife']
print(include_types)

nightlife_c = PlaceTextSearchClient()
nightlife_pids = set()
for query in include_types:
    print(f"########### Starting search for: {query} ################")
    for poly in tqdm(polygon_list):
        nightlife_pids = nightlife_pids.union(
        nightlife_c.paginate_requests_placeid_only(
            query.replace("_", " "), poly, page_token=None, included_type=query)
        )
    print(len(nightlife_pids))

pd.DataFrame({"place_ids": list(nightlife_pids)}).assign(place_type="nightlife").to_parquet(f"{save_places_dir}/{now_ts}_nightlife.parquet")

['comedy_club', 'concert_hall', 'dance_hall', 'night_club', 'philharmonic_hall', 'live_music_venue']
########### Starting search for: comedy_club ################


100%|██████████| 1113/1113 [04:49<00:00,  3.85it/s]


593
########### Starting search for: concert_hall ################


100%|██████████| 1113/1113 [04:51<00:00,  3.82it/s]


854
########### Starting search for: dance_hall ################


...

3091
########### Starting search for: philharmonic_hall ################


100%|██████████| 1113/1113 [04:29<00:00,  4.13it/s]


3093
########### Starting search for: live_music_venue ################


100%|██████████| 1113/1113 [06:11<00:00,  3.00it/s] 

5420


In [ ]:
include_types = gmaps_types['hybrid']
print(include_types)

hybrid_c = PlaceTextSearchClient()
hybrid_pids = set()
for query in include_types:
    print(f"########### Starting search for: {query} ################")
    for poly in tqdm(polygon_list):
        hybrid_pids = hybrid_pids.union(
        hybrid_c.paginate_requests_placeid_only(
            query.replace("_", " "), poly, page_token=None, included_type=query)
        )
    print(len(hybrid_pids))

pd.DataFrame({"place_ids": list(hybrid_pids)}).assign(place_type="hybrid").to_parquet(f"{save_places_dir}/{now_ts}_hybrid.parquet")

['community_center', 'cultural_center', 'event_venue', 'convention_center', 'botanical_garden', 'national_park', 'state_park']
########### Starting search for: event_venue ################


100%|██████████| 1113/1113 [11:27<00:00,  1.62it/s]


26059
########### Starting search for: convention_center ################


100%|██████████| 1113/1113 [04:49<00:00,  3.85it/s]


26093
########### Starting search for: botanical_garden ################


...

26307
########### Starting search for: national_park ################


100%|██████████| 1113/1113 [05:09<00:00,  3.60it/s]


26839
########### Starting search for: state_park ################


100%|██████████| 1113/1113 [04:51<00:00,  3.82it/s]

26853


### Send Place Details API requests

In [15]:
pilot_pids = pd.read_parquet(
    [os.path.join(save_places_dir,fn) for fn in os.listdir(save_places_dir) if fn.startswith("28-02-2026_101817")]
)

pilot_pids.info()
print(pilot_pids.nunique())

rest_pids = pd.read_parquet(
    [os.path.join(save_places_dir,fn) for fn in os.listdir(save_places_dir) if fn.startswith("28-02-2026_121413")]
)

rest_pids.info()
print(rest_pids.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4058 entries, 0 to 4057
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   place_ids   4058 non-null   object
 1   place_type  4058 non-null   object
dtypes: object(2)
memory usage: 63.5+ KB
place_ids     3933
place_type       6
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37549 entries, 0 to 37548
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   place_ids   37549 non-null  object
 1   place_type  37549 non-null  object
dtypes: object(2)
memory usage: 586.8+ KB
place_ids     36550
place_type        6
dtype: int64


In [16]:
rerun_pids = pd.read_parquet(
    [os.path.join(save_places_dir,fn) for fn in os.listdir(save_places_dir) if fn.startswith("23-04-2026_204939")]
)

rerun_pids.info()
print(rerun_pids.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84934 entries, 0 to 84933
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   place_ids   84934 non-null  object
 1   place_type  84934 non-null  object
dtypes: object(2)
memory usage: 1.3+ MB
place_ids     80331
place_type        7
dtype: int64


In [30]:
details_completed = pd.read_parquet([os.path.join(save_places_dir,fn) for fn in os.listdir(save_places_dir) 
                                     if (fn.find("_pid_details")>0 or fn.find("geocode_results")>=0)])
details_completed = set(details_completed.index)
print(len(details_completed), "extracted")

to_get_details_pids = rerun_pids.loc[~rerun_pids.place_ids.isin(details_completed)].drop_duplicates(subset='place_ids')
print(len(to_get_details_pids), "remaining")

59484 extracted
22762 remaining


In [31]:
pro_details = pd.read_parquet([os.path.join(save_places_dir,fn) for fn in os.listdir(save_places_dir) if fn.find("_pid_details_pro")>0])
pro_details = set(pro_details.index)
print(len(pro_details), "with pro details")

13554 with pro details


In [32]:
# sample_check_pids = pd.concat([rerun_pids.sample(frac=0.05),
#                                rerun_pids.groupby("place_type").sample(250)],
#                                ignore_index=True
# ).drop_duplicates(subset='place_ids')
# sample_check_pids['pro_details_extracted'] = sample_check_pids.place_ids.isin(pro_details)
# print( sample_check_pids.place_type.value_counts() )
# print( sample_check_pids.pro_details_extracted.value_counts() )
# len( sample_check_pids )

# sample_check_pids.to_parquet("../data/sample_check_pids.parquet")

sample_check_pids = pd.read_parquet("../data/sample_check_pids.parquet")
sample_check_pids.info()
sample_check_pids.place_ids.isin(to_get_details_pids).value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 5862 entries, 0 to 5996
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   place_ids              5862 non-null   object
 1   place_type             5862 non-null   object
 2   pro_details_extracted  5862 non-null   bool  
dtypes: bool(1), object(2)
memory usage: 143.1+ KB


place_ids
False    5862
Name: count, dtype: int64

In [33]:
from scripts.google_places_search import PlaceDetailsClient

pd_c = PlaceDetailsClient()

ess_fm = """formattedAddress
location
plusCode
postalAddress
shortFormattedAddress
types""".split("\n")                            #essential fieldmasks -- cap at 10,000 free call per month

pro_fm = """accessibilityOptions
businessStatus
containingPlaces
displayName
googleMapsLinks
googleMapsUri
iconBackgroundColor
iconMaskBaseUri
primaryType
primaryTypeDisplayName
pureServiceAreaBusiness
subDestinations
timeZone
utcOffsetMinutes
""".split("\n")                                 #pro fieldmasks -- cap at 5,000 free call per month

ent_fm = """currentOpeningHours
currentSecondaryOpeningHours
internationalPhoneNumber
nationalPhoneNumber
priceLevel
priceRange
rating
regularOpeningHours
regularSecondaryOpeningHours
userRatingCount
websiteUri""".split("\n")                        #enterprise fieldmasks -- cap at 1,000 free call per month

fieldmasks = set(ess_fm).union(['displayName', 'primaryType', 'primaryTypeDisplayName', 'googleMapsUri', 'googleMapsLinks', 'businessStatus', 'containingPlaces', 'subDestinations', 'accessibilityOptions'])
fieldmasks

{'accessibilityOptions',
 'businessStatus',
 'containingPlaces',
 'displayName',
 'formattedAddress',
 'googleMapsLinks',
 'googleMapsUri',
 'location',
 'plusCode',
 'postalAddress',
 'primaryType',
 'primaryTypeDisplayName',
 'shortFormattedAddress',
 'subDestinations',
 'types'}

In [34]:
place_details = {}

get_pro_details = to_get_details_pids.sample(4795).place_ids.unique() # should be 5000 but i ran 205 before this

for pid in tqdm(get_pro_details):  
    try:
        place_details[pid] = pd_c.request_pid(pid, fieldmasks).json()
    except Exception as e:
        print("Failed to get:", pid)
        print(e)

print(len(place_details))

place_details_df = pd.DataFrame(place_details).T
place_details_df.to_parquet(f"{save_places_dir}/full_pid_details_pro_may.parquet")

  4%|▍         | 203/4795 [00:44<14:12,  5.38it/s]

404 Client Error: Not Found for url: https://places.googleapis.com/v1/places/ChIJGwUzXAZ5cUgRhMjOrvrPNrQ
Failed to get: ChIJGwUzXAZ5cUgRhMjOrvrPNrQ
'str' object has no attribute 'json'


 20%|█▉        | 941/4795 [03:14<11:05,  5.79it/s]

404 Client Error: Not Found for url: https://places.googleapis.com/v1/places/ChIJ4-HiAABNekgRhzC5tbK8-yE
Failed to get: ChIJ4-HiAABNekgRhzC5tbK8-yE
'str' object has no attribute 'json'


 25%|██▌       | 1218/4795 [04:09<11:16,  5.29it/s]

...

404 Client Error: Not Found for url: https://places.googleapis.com/v1/places/ChIJpdM2fQCbd0gRAvC68SQ7soI
Failed to get: ChIJpdM2fQCbd0gRAvC68SQ7soI
'str' object has no attribute 'json'


 99%|█████████▊| 4734/4795 [16:12<00:10,  5.55it/s]

404 Client Error: Not Found for url: https://places.googleapis.com/v1/places/ChIJ70yANQAvdEgR2nzAtyukAG0
Failed to get: ChIJ70yANQAvdEgR2nzAtyukAG0
'str' object has no attribute 'json'


100%|██████████| 4795/4795 [16:25<00:00,  4.87it/s]


4964


In [38]:
get_ess_details = set(to_get_details_pids.place_ids) - set(get_pro_details)
print("remaining to get:", len(get_ess_details))

get_ess_details = list(get_ess_details)[:10000]

remaining to get: 17967


In [39]:
place_details_limited = {}

fieldmasks = set(ess_fm)#.union(['displayName', 'primaryType', 'primaryTypeDisplayName', 'googleMapsUri', 'googleMapsLinks', 'businessStatus', 'containingPlaces', 'subDestinations', 'accessibilityOptions'])
print(fieldmasks)

pd_ess_c = PlaceDetailsClient()

for pid in tqdm(get_ess_details):  # run 10K within free usage
    try:
        place_details_limited[pid] = pd_ess_c.request_pid(pid, fieldmasks).json()
    except Exception as e:
        print("Failed to get:", pid)
        print(e)

print(len(place_details_limited))

place_details_lim_df = pd.DataFrame(place_details_limited).T
place_details_lim_df.to_parquet(f"{save_places_dir}/full_pid_details_ess_may.parquet")

{'formattedAddress', 'postalAddress', 'types', 'location', 'plusCode', 'shortFormattedAddress'}


  1%|          | 67/10000 [00:16<36:48,  4.50it/s]  

404 Client Error: Not Found for url: https://places.googleapis.com/v1/places/ChIJteRFHAB1e0gRe9ynG0_fWHs
Failed to get: ChIJteRFHAB1e0gRe9ynG0_fWHs
'str' object has no attribute 'json'


  4%|▎         | 372/10000 [01:23<35:12,  4.56it/s]  

404 Client Error: Not Found for url: https://places.googleapis.com/v1/places/ChIJiw2xLOFd30cRcMO2WtlD400
Failed to get: ChIJiw2xLOFd30cRcMO2WtlD400
'str' object has no attribute 'json'


...

404 Client Error: Not Found for url: https://places.googleapis.com/v1/places/ChIJJwctegBxfkgR-7gKCI4rYCM
Failed to get: ChIJJwctegBxfkgR-7gKCI4rYCM
'str' object has no attribute 'json'


 96%|█████████▌| 9610/10000 [34:07<01:18,  4.99it/s]

404 Client Error: Not Found for url: https://places.googleapis.com/v1/places/ChIJAVjmTABNckgRiP_FiXVwlyU
Failed to get: ChIJAVjmTABNckgRiP_FiXVwlyU
'str' object has no attribute 'json'


100%|██████████| 10000/10000 [35:24<00:00,  4.71it/s]


9985


In [40]:
get_geocode_details = set(to_get_details_pids.place_ids) - set(get_pro_details) - set(get_ess_details)
print("remaining to get:", len(get_geocode_details))

get_geocode_details = list(get_geocode_details)[:10000]

remaining to get: 7967


In [42]:
from scripts.google_places_search import GeocodingClient

geocode_c = GeocodingClient()
geocode_results = {}

for pid in tqdm(get_geocode_details):  # run 10K within free usage
    try:
        geocode_results[pid] = geocode_c.request_pid(pid, fieldmasks).json()
    except Exception as e:
        print("Failed to get:", pid)
        print(e)

print(len(geocode_results))

geocode_results_df = pd.DataFrame(geocode_results).T
geocode_results_df.to_parquet(f"{save_places_dir}/geocode_results_may.parquet")

  2%|▏         | 152/7967 [00:21<16:22,  7.96it/s]

404 Client Error: Not Found for url: https://geocode.googleapis.com/v4/geocode/places/ChIJrctPr-a3d0gRR-r1sie_T6w
Failed to get: ChIJrctPr-a3d0gRR-r1sie_T6w
'str' object has no attribute 'json'


  5%|▌         | 420/7967 [00:59<17:45,  7.08it/s]

404 Client Error: Not Found for url: https://geocode.googleapis.com/v4/geocode/places/ChIJq3iEXgANdkgR7wfZD7jMVJo
Failed to get: ChIJq3iEXgANdkgR7wfZD7jMVJo
'str' object has no attribute 'json'


  5%|▌         | 429/7967 [01:01<19:16,  6.52it/s]

...

404 Client Error: Not Found for url: https://geocode.googleapis.com/v4/geocode/places/ChIJOTJiLQCxe0gRr17T9inn5TY
Failed to get: ChIJOTJiLQCxe0gRr17T9inn5TY
'str' object has no attribute 'json'


 74%|███████▍  | 5893/7967 [15:22<07:13,  4.79it/s]

404 Client Error: Not Found for url: https://geocode.googleapis.com/v4/geocode/places/ChIJx9NIfgD7fUgR3w0F6C1CE_U
Failed to get: ChIJx9NIfgD7fUgR3w0F6C1CE_U
'str' object has no attribute 'json'


100%|██████████| 7967/7967 [20:46<00:00,  6.39it/s]


7956
